In [26]:

import os
import pdfplumber
import pandas as pd

In [71]:

all_dataframes = []
base_path = r"C:\Users\nicho\pdfs"
for file_name in os.listdir(base_path):
    if file_name.lower().endswith(".pdf"):
        file_path = os.path.join(base_path, file_name)

        with pdfplumber.open(file_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                tables = page.extract_tables()

                for table in tables:
                    if table:  # make sure table is not empty
                        df = pd.DataFrame(table)

                        # Optional: add metadata columns
                        df["source_file"] = file_name
                        df["page"] = page_num + 1

                        all_dataframes.append(df)
print(all_dataframes[1].head())

KeyboardInterrupt: 

In [48]:
print(all_dataframes[20])

      0                               1    2    3    4     5      6    7    8  \
0  Rank                             NOC  Men  NaN  NaN   NaN  Women  NaN  NaN   
1   NaN                             NaN    G    S    B  Tot.      G    S    B   
2     1                   GER - Germany    2    2    1     5      1    2        
3     2  USA - United States of America                           1         2   
4     3               SUI - Switzerland              1     1                    
5                                Total:    2    2    2     6      2    2    2   

      9     10   11   12    13               14  \
0   NaN  Total  NaN  NaN   NaN  Rank\nby\nTotal   
1  Tot.      G    S    B  Tot.              NaN   
2     3      3    4    1     8                1   
3     3      1         2     3                2   
4                      1     1                3   
5     6      4    4    4    12                    

                                       source_file  page  
0  BOB---------

In [50]:

# Remove empty dataframes and those that are mostly NaN
cleaned_dataframes = []

for df in all_dataframes:
    # Skip if dataframe is empty
    if df.empty:
        continue
    
    # Skip if more than 50% of values are NaN
    if df.isna().sum().sum() / (len(df) * len(df.columns)) > 0.5:
        continue
    
    # Skip if all values in the first row are NaN
    if df.iloc[0].isna().all():
        continue
    
    cleaned_dataframes.append(df)

all_dataframes = cleaned_dataframes
print(f"Original: {len(all_dataframes)} dataframes | Cleaned: {len(cleaned_dataframes)} dataframes")



Original: 187 dataframes | Cleaned: 187 dataframes
                 0    1      2      3  \
0              NOC  Men  Women  Total   
1        AIN - AIN    1      2      3   
2    ALB - Albania    1      3      4   
3    AND - Andorra    2      3      5   
4  ARG - Argentina    1      2      3   

                                       source_file  page  
0  ALP-------------------------------__C30_5.0.pdf     1  
1  ALP-------------------------------__C30_5.0.pdf     1  
2  ALP-------------------------------__C30_5.0.pdf     1  
3  ALP-------------------------------__C30_5.0.pdf     1  
4  ALP-------------------------------__C30_5.0.pdf     1  


In [70]:
# Group dataframes by their columns and concatenate those with matching columns
grouped = {}

for df in all_dataframes:
    # Create a hashable key from the column names
    col_key = tuple(df.columns)
    
    if col_key not in grouped:
        grouped[col_key] = []
    grouped[col_key].append(df)

# Concatenate dataframes with matching columns
condensed_dataframes = []
for col_key, dfs in grouped.items():
    if len(dfs) > 0:
        concatenated_df = pd.concat(dfs, ignore_index=True)
        condensed_dataframes.append(concatenated_df)

all_dataframes = condensed_dataframes
print(f"Condensed from 187 dataframes to {len(all_dataframes)} dataframes")


Condensed from 187 dataframes to 13 dataframes


,0,1,2,3,4,5,6,7,8,9,10,11,12,source_file,page
0,FIS NOC 1.8km 4.9km 8.6km Finish FIS\nRank Bib...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CCSM10KMIS------------FNL-000100--__C73A_1.0.pdf,1
1,1,44,3422819,KLAEBO Johannes Hoesflot NOR,3:34.1,12,10:32.6,2,17:53.6,2,20:36.2,,0.00,CCSM10KMIS------------FNL-000100--__C73A_1.0.pdf,1
2,2,46,3190634,DESLOGES Mathis FRA,3:26.6,1,10:34.3,3,17:55.9,3,20:41.1,+4.9,3.17,CCSM10KMIS------------FNL-000100--__C73A_1.0.pdf,1
3,3,62,3424757,HEDEGART Einar NOR,3:27.8,2,10:27.4,1,17:50.8,1,20:50.2,+14.0,9.06,CCSM10KMIS------------FNL-000100--__C73A_1.0.pdf,1
4,4,58,3423264,AMUNDSEN Harald Oestberg NOR,3:28.0,3,10:39.1,4,18:06.7,4,21:00.2,+24.0,15.53,CCSM10KMIS------------FNL-000100--__C73A_1.0.pdf,1


In [73]:
import os

# Create the project_csvs folder if it doesn't exist
base_path = r"C:\Users\nicho"
csv_folder = os.path.join(base_path, 'project_csvs')
os.makedirs(csv_folder, exist_ok=True)

# Save each dataframe as a CSV file
for i, df in enumerate(all_dataframes):
    print(all_dataframes[i].head())
    csv_filename = os.path.join(csv_folder, f'dataframe_{i}.csv')
    df.to_csv(csv_filename, index=False)
    

print(f"Saved {len(all_dataframes)} dataframes to {csv_folder}")

             0                                      source_file  page
0      REVISED  ALP-------------------------------__C30_5.0.pdf     1
1  10 FEB 7:30  ALP-------------------------------__C30_5.0.pdf     1
                 0    1      2      3  \
0              NOC  Men  Women  Total   
1        AIN - AIN    1      2      3   
2    ALB - Albania    1      3      4   
3    AND - Andorra    2      3      5   
4  ARG - Argentina    1      2      3   

                                       source_file  page  
0  ALP-------------------------------__C30_5.0.pdf     1  
1  ALP-------------------------------__C30_5.0.pdf     1  
2  ALP-------------------------------__C30_5.0.pdf     1  
3  ALP-------------------------------__C30_5.0.pdf     1  
4  ALP-------------------------------__C30_5.0.pdf     1  
             0                                      source_file  page
0      REVISED  ALP-------------------------------__C30_5.0.pdf     2
1  10 FEB 7:30  ALP------------------------------